In [ ]:

# --- INSTALL ---
!pip install -q transformers datasets accelerate peft trl bitsandbytes gradio

# --- IMPORTS ---
import torch
import gc
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

# --- LOAD DATASET ---
dataset = load_dataset("gretelai/synthetic_text_to_sql")
print(f"Train: {len(dataset['train'])}, Test: {len(dataset['test'])}")

# --- LOAD MODEL (4-bit quantized) ---
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False
print(f"Model loaded: {model.get_memory_footprint()/1e9:.1f} GB")

# --- HELPER FUNCTIONS ---
def create_prompt(sample):
    return f"""[INST] You are a helpful SQL assistant. Given the following database schema and a natural language question, generate the correct SQL query and explain what it does.

### Database Schema:
{sample["sql_context"]}

### Question:
{sample["sql_prompt"]}

### Response:
[/INST]"""

def generate_response(prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(out[0], skip_special_tokens=True)
    if "[/INST]" in response:
        response = response.split("[/INST]")[-1].strip()
    return response

# --- TEST BEFORE FINE-TUNING ---
test_samples = [dataset["test"][i] for i in [0, 10, 50]]

print("\n" + "=" * 60)
print("BEFORE FINE-TUNING")
print("=" * 60)

before_results = []
for i, sample in enumerate(test_samples):
    prompt = create_prompt(sample)
    response = generate_response(prompt)
    print(f"\n--- Example {i+1} ---")
    print(f"Question: {sample['sql_prompt']}")
    print(f"Expected: {sample['sql']}")
    print(f"Model:    {response[:300]}")
    before_results.append({
        "question": sample["sql_prompt"],
        "expected": sample["sql"],
        "output": response,
    })

# --- CLEAR MEMORY BEFORE TRAINING ---
gc.collect()
torch.cuda.empty_cache()

# --- FORMAT TRAINING DATA ---
def format_training_sample(sample):
    return f"""[INST] You are a helpful SQL assistant. Given the following database schema and a natural language question, generate the correct SQL query and explain what it does.

### Database Schema:
{sample["sql_context"]}

### Question:
{sample["sql_prompt"]}

### Response:
[/INST]
**SQL Query:**
{sample["sql"]}

**Explanation:**
{sample["sql_explanation"]}"""

train_data = dataset["train"].shuffle(seed=42).select(range(1000))
train_data = train_data.map(lambda s: {"formatted_text": format_training_sample(s)})
formatted_dataset = train_data.map(
    lambda x: {"text": x["formatted_text"]},
    remove_columns=train_data.column_names,
)
print(f"\nTraining samples: {len(formatted_dataset)}")

# --- CONFIGURE LORA + TRAINING ---
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

training_args = TrainingArguments(
    output_dir="./mistral-sql-lora",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_strategy="no",
    optim="paged_adamw_8bit",
    warmup_steps=10,
    max_grad_norm=0.3,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=training_args,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")
print("Starting training...\n")

# --- TRAIN ---
trainer.train()
print("\n✅ Training complete!")

# --- TEST AFTER FINE-TUNING ---
model.config.use_cache = True

print("\n" + "=" * 60)
print("AFTER FINE-TUNING")
print("=" * 60)

after_results = []
for i, sample in enumerate(test_samples):
    prompt = create_prompt(sample)
    response = generate_response(prompt)
    print(f"\n--- Example {i+1} ---")
    print(f"Question: {sample['sql_prompt']}")
    print(f"Expected: {sample['sql']}")
    print(f"Model:    {response[:300]}")
    after_results.append({
        "question": sample["sql_prompt"],
        "expected": sample["sql"],
        "output": response,
    })

# --- BEFORE vs AFTER COMPARISON ---
print("\n" + "=" * 60)
print("BEFORE vs AFTER COMPARISON")
print("=" * 60)
for i in range(len(test_samples)):
    print(f"\n{'='*60}")
    print(f"Example {i+1}: {before_results[i]['question']}")
    print(f"{'='*60}")
    print(f"✅ Expected:\n{before_results[i]['expected']}")
    print(f"\n❌ Before fine-tuning:\n{before_results[i]['output'][:300]}")
    print(f"\n🟢 After fine-tuning:\n{after_results[i]['output'][:300]}")

In [ ]:
import re
import gradio as gr
import pandas as pd

def extract_sql_from_response(response):
    if "**SQL Query:**" in response:
        sql = response.split("**SQL Query:**")[1]
        if "**Explanation:**" in sql:
            sql = sql.split("**Explanation:**")[0]
        return sql.strip()
    if "```sql" in response:
        sql = response.split("```sql")[1]
        if "```" in sql:
            sql = sql.split("```")[0]
        return sql.strip()
    return response.strip()

def simple_sql_match(predicted, expected):
    def normalize(s):
        s = s.lower().strip().rstrip(";")
        s = re.sub(r'\s+', ' ', s)
        return s
    return normalize(predicted) == normalize(expected)

def is_cutoff(response):
    # A response is considered cut off if it has no ending punctuation or is under 20 chars
    return len(response.strip()) < 20 or not response.strip()[-1] in '.;)'

def format_consistent(response):
    return "**SQL Query:**" in response and "**Explanation:**" in response

# --- EVALUATE BEFORE AND AFTER ON SAME 20 SAMPLES ---
eval_indices = list(range(0, 200, 10))

correct_before = 0
correct_after = 0
before_format = 0
after_format = 0
before_cutoffs = 0
after_cutoffs = 0
results_table = []

print("Evaluating 20 test samples (before and after)...")
print("=" * 60)

# Temporarily disable LoRA to get before scores
model.disable_adapter_layers()
model.config.use_cache = True

for idx in eval_indices:
    sample = dataset["test"][idx]
    prompt = create_prompt(sample)
    expected = sample["sql"]

    # --- BEFORE ---
    before_response = generate_response(prompt)
    before_sql = extract_sql_from_response(before_response)
    before_match = simple_sql_match(before_sql, expected)
    if before_match:
        correct_before += 1
    if format_consistent(before_response):
        before_format += 1
    if is_cutoff(before_response):
        before_cutoffs += 1

    results_table.append({
        "question": sample["sql_prompt"][:60],
        "before_match": "✅" if before_match else "❌",
        "after_match": None,  # fill in next loop
        "idx": idx
    })

# Re-enable LoRA for after scores
model.enable_adapter_layers()

for i, idx in enumerate(eval_indices):
    sample = dataset["test"][idx]
    prompt = create_prompt(sample)
    expected = sample["sql"]

    # --- AFTER ---
    after_response = generate_response(prompt)
    after_sql = extract_sql_from_response(after_response)
    after_match = simple_sql_match(after_sql, expected)
    if after_match:
        correct_after += 1
    if format_consistent(after_response):
        after_format += 1
    if is_cutoff(after_response):
        after_cutoffs += 1

    results_table[i]["after_match"] = "✅" if after_match else "❌"

# --- PRINT COMPARISON TABLE ---
print(f"\n{'='*60}")
print("BEFORE vs AFTER — QUANTITATIVE COMPARISON")
print(f"{'='*60}")
print(f"{'Metric':<35} {'Before':>8} {'After':>8} {'Change':>8}")
print(f"{'-'*60}")
print(f"{'Exact match (20 samples)':<35} {correct_before}/20{' ':>4} {correct_after}/20{' ':>4} {'+' if correct_after >= correct_before else ''}{correct_after - correct_before:+d} pts")
print(f"{'Exact match %':<35} {correct_before/20*100:.0f}%{' ':>5} {correct_after/20*100:.0f}%{' ':>5} {(correct_after - correct_before)/20*100:+.0f}%")
print(f"{'Format consistent (SQL+Exp)':<35} {before_format}/20{' ':>4} {after_format}/20{' ':>4} {after_format - before_format:+d}")
print(f"{'Cut-off outputs':<35} {before_cutoffs}/20{' ':>4} {after_cutoffs}/20{' ':>4} {after_cutoffs - before_cutoffs:+d}")
print(f"{'='*60}")

# --- PER-SAMPLE TABLE ---
df = pd.DataFrame([{"Question": r["question"], "Before": r["before_match"], "After": r["after_match"]} for r in results_table])
print("\nPer-sample results:")
print(df.to_string(index=False))

# --- GRADIO INTERFACE ---
def query_model(schema, question):
    sample = {"sql_context": schema, "sql_prompt": question}
    prompt = create_prompt(sample)
    response = generate_response(prompt, max_new_tokens=300)
    return response

demo = gr.Interface(
    fn=query_model,
    inputs=[
        gr.Textbox(label="Database Schema", placeholder="CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);", lines=4),
        gr.Textbox(label="Your Question", placeholder="Show me all employees in the sales department", lines=2),
    ],
    outputs=gr.Textbox(label="Generated SQL + Explanation", lines=8),
    title="🔍 Text-to-SQL — Fine-tuned Mistral-7B",
    description="Enter a database schema and ask a question in plain English. The fine-tuned model will generate the SQL query and explain it.",
    examples=[
        ["CREATE TABLE employees (id INT, name TEXT, department TEXT, salary REAL);", "What is the average salary of employees in each department?"],
        ["CREATE TABLE orders (order_id INT, customer_id INT, product TEXT, amount REAL, order_date DATE);", "Find the total amount spent by each customer in 2024"],
        ["CREATE TABLE students (id INT, name TEXT, grade INT, city TEXT); CREATE TABLE scores (student_id INT, subject TEXT, score INT);", "What is the average score of students from New York?"],
    ],
)

print("\nLaunching Gradio interface...")
demo.launch(share=True)

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=query_model,
    inputs=[
        gr.Textbox(
            label="Database Schema",
            placeholder="CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);",
            lines=4,
        ),
        gr.Textbox(
            label="Your Question",
            placeholder="Show me all employees in the sales department",
            lines=2,
        ),
    ],
    outputs=gr.Textbox(label="Generated SQL + Explanation", lines=8),
    title="🔍 Text-to-SQL — Fine-tuned Mistral-7B",
    description="Enter a database schema and ask a question in plain English. The fine-tuned model will generate the SQL query and explain it.",
    examples=[
        [
            "CREATE TABLE employees (id INT, name TEXT, department TEXT, salary REAL);",
            "What is the average salary of employees in each department?"
        ],
        [
            "CREATE TABLE orders (order_id INT, customer_id INT, product TEXT, amount REAL, order_date DATE);",
            "Find the total amount spent by each customer in 2024"
        ],
        [
            "CREATE TABLE students (id INT, name TEXT, grade INT, city TEXT); CREATE TABLE scores (student_id INT, subject TEXT, score INT);",
            "What is the average score of students from New York?"
        ],
    ],
)

print("Launching Gradio interface...")
demo.launch(share=True)